# Drishti — End-to-End App Test (on Colab)

Runs the **real `app/` code** — router, engines, guardrail, translation, speech — against
real photos on Colab. This is the Phase-1 exit criterion, executed without installing
anything locally.

Everything in `app/` is unit-tested with fakes (139 tests). What has never happened is a
real model loading through it. That is what this notebook checks.

### Why the repo has to be fetched

The Colab VS Code extension runs *cells* on a Colab machine; it does not copy your project
there. `app/` therefore does not exist on the runtime until §1 fetches it — via `git clone`
(recommended) or a zip upload.

### Why §4 runs OCR in a subprocess

Medicine mode killed this notebook repeatedly. Three separate causes, found in order:

1. **Out of RAM.** OCR ran in the kernel, which also held transformers, torch and
   TensorFlow. Colab reported *"session crashed after using all available RAM"*, taking
   every cell above it with it.
2. **`max_side=1600` never downscaled anything.** The engine resizes only when the longest
   side *exceeds* `max_side`, and the fixtures are exactly 1600×1204 — so document
   unwarping ran on the full 1.9 MP image. §4 passes `--max-side 1280`, which reads the
   drug name, expiry and MRP correctly and is measurably faster. (It was briefly reverted
   to 1600 on the belief that 1280 had cost an expiry date; that was wrong — `DEC-034`.)
3. **A segfault importing `libpaddle`** — the real one. `paddleocr` defers loading paddle to
   `paddlex`, which imports torch and TensorFlow *first*; paddle then initializes into a
   process that already owns the glog/gflags/OpenMP symbols it needs, and dies. The fix is
   import order: **paddle first, into a clean process.**

Cause 3 is a load-time crash, which is why `--fast` and `--max-side` made no difference to
it — both are inference-time knobs. It was only findable because the subprocess enabled
`faulthandler` and ran unbuffered; the C stack named the library in one run.

So §4 writes `/content/ocr_phase.py` and runs it separately. Consequences:

- **no manual restart is needed** — run the notebook straight through
- the OS reclaims the OCR memory on exit, so §6–§7 start clean
- a crash kills the child and prints a decoded signal, instead of your session

**Runtime → Change runtime type → T4 GPU** before running. §7's VLM is very slow on CPU
(the last run had no GPU — `nvidia-smi: command not found`).

In [1]:
import platform, os
print(platform.system(), platform.release())
print('hostname:', platform.node())
print('cwd:', os.getcwd())
!nvidia-smi --query-gpu=name --format=csv,noheader


Linux 6.6.122+
hostname: ef1892c9677e
cwd: /content
Tesla T4


## 1. Get the project onto the runtime

The Colab VS Code extension runs *cells* on Colab hardware but leaves your files on the
local disk, so `app/` does not exist on the runtime until we put it there.

> **`google.colab.files.upload()` does not work from VS Code.** It is a browser widget: the
> HTML renders, the JavaScript bridge that picks the file never loads, and the cell hangs
> until you interrupt it. Use one of the two paths below instead.

### Path A — git clone (recommended)

Push the project to GitHub once, then set `REPO_URL` below. Every later run is a single cell
that always pulls current code, and the repo ends up backed up and shareable with your guide.

```powershell
git remote add origin https://github.com/<you>/drishti.git
git push -u origin main
```

### Path B — run this notebook in the Colab browser

Open [colab.research.google.com](https://colab.research.google.com), upload this notebook,
and `files.upload()` behaves normally. Zero setup, but you lose the VS Code editor.

Sample photos are committed under `data/samples/`, so **no image upload is needed either
way** — that failure mode is gone entirely.

In [2]:
import os

# Set before any framework import -- see DEC-006.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import shutil, subprocess, sys, zipfile
from pathlib import Path

REPO_URL = 'https://github.com/DevGurav/Drishti.git'
REFRESH = True         # re-fetch every run; set False only to keep a hand-edited runtime
PROJECT = Path('/content/drishti')
WORKDIR = Path('/content')

CLONE_HELP = """
If the repository is PRIVATE, clone with a token:
  GitHub -> Settings -> Developer settings -> Personal access tokens
  -> Fine-grained token, Repository access: this repo, Contents: Read
  REPO_URL = 'https://<TOKEN>@github.com/DevGurav/Drishti.git'

Otherwise: make the repo public, or use Path B (Colab browser + zip upload).
"""


def _find_project_root(start: Path):
    """Locate the folder holding app/router.py, however the archive nested it."""
    if (start / 'app' / 'router.py').exists():
        return start
    for marker in start.glob('*/app/router.py'):
        return marker.parents[1]
    return None


# Step out of PROJECT before deleting it. A re-run leaves the process cwd inside the
# project, and deleting the directory you are standing in makes every later subprocess
# fail with "unable to read current working directory" -- git exits 128.
os.chdir(WORKDIR)

# A stale copy is the failure that actually bites: the runtime keeps whatever was fetched
# first, so pushing new code changes nothing here and the error surfaces somewhere else.
if REFRESH:
    for stale in (PROJECT, WORKDIR / '_clone', WORKDIR / '_unpack'):
        shutil.rmtree(stale, ignore_errors=True)

if not (PROJECT / 'app' / 'router.py').exists():
    if REPO_URL:
        clone = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR / '_clone')],
            capture_output=True, text=True)
        if clone.returncode != 0:
            # Print git's own message. check=True hides stderr, and the cause is usually
            # only visible there (private repo, typo, auth).
            print(f'git clone failed (exit {clone.returncode}):')
            print(clone.stderr.strip())
            print(CLONE_HELP)
            raise SystemExit('clone failed -- see the message above')
        found = _find_project_root(WORKDIR / '_clone')
        if found is None:
            raise SystemExit('app/router.py not found in the cloned repo.')
        shutil.move(str(found), str(PROJECT))
    else:
        # Path B only -- this widget works in the Colab browser, never from VS Code.
        try:
            from google.colab import files
        except ImportError:
            raise SystemExit('Not running on Colab. Set REPO_URL above.')
        print('Upload drishti.zip  (Colab browser only; from VS Code set REPO_URL instead)')
        up = files.upload()
        with zipfile.ZipFile(next(iter(up))) as z:
            z.extractall(WORKDIR / '_unpack')
        found = _find_project_root(WORKDIR / '_unpack')
        if found is None:
            raise SystemExit('app/router.py not in the archive -- did you zip the drishti '
                             'folder itself?')
        shutil.move(str(found), str(PROJECT))

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

samples = sorted((PROJECT / 'data' / 'samples').glob('*.jpg'))
head = subprocess.run(['git', '-C', str(PROJECT), 'log', '--oneline', '-1'],
                      capture_output=True, text=True).stdout.strip()

print('project root :', PROJECT)
print('fetched HEAD :', head or '(not a git checkout)')
print('modes        :', sorted(p.stem for p in (PROJECT / 'app' / 'modes').glob('[!_]*.py')))
print('sample images:', [p.name for p in samples] or 'NONE')

# Fail here, naming the real cause, rather than three cells later with a missing file.
if not samples:
    print('data/samples/ is empty, so the fetched code is out of date.')
    print('Most likely your local commits have not been pushed yet:')
    print('    git push')
    print('then re-run this cell (REFRESH=True forces a fresh clone).')
    raise SystemExit('stale code -- see the message above')

project root : /content/drishti
fetched HEAD : 5c651f9 docs: record the expiry-date bug and the paddle/torch split
modes        : ['ask', 'currency', 'medicine', 'read', 'scene']
sample images: ['strip_paracip.jpg', 'strip_partial.jpg']


In [3]:
# The suite needs no models, so a pass here proves the upload is complete and importable
# before we spend minutes downloading weights.
!python -m unittest discover -s tests -t . 2>&1 | tail -4

----------------------------------------------------------------------
Ran 124 tests in 0.126s

OK


## 2. Install engines

Weights are **not** downloaded here — every engine loads lazily on first use, so each mode
below pays only for what it needs.

> **Ignore this warning if you see it:** `gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0,
> but you have huggingface-hub 0.36.2 which is incompatible.` Colab's base image ships
> `gradio` pre-installed; installing `transformers`/`IndicTransToolkit` pins an older
> `huggingface-hub` than gradio declares it wants. Nothing in this project imports `gradio`,
> so the conflict is real but inert.

In [ ]:
# Install only -- deliberately no `import transformers` here.
#
# An earlier version verified the version by importing it in this cell. That put
# transformers, torch and (because Colab preinstalls it) TensorFlow into the kernel before
# §4 ran, costing 1-2 GB of RAM that the OCR phase has no use for. §4 then ran out of
# memory. The version check now lives in the torch-phase bootstrap, after OCR is done.
#
# transformers<5 is required by IndicTransToolkit, which imports PreTrainedTokenizerBase
# from transformers.tokenization_utils -- removed in v5. SmolVLM (§7) is not the reason for
# the pin: it already runs on 4.x, proven in notebook 00.
%pip install -q "transformers<5" paddlepaddle paddleocr IndicTransToolkit
print('installed -- transformers stays unimported until §6')

## 3. Choose test photos

Committed fixtures in `data/samples/` are used by default, so nothing needs uploading.

`strip_paracip.jpg` is the read that produced 55 OCR lines including the drug name,
`MFD.MAY 25 EXP.APR.28` and `Rs.10.30`. `strip_partial.jpg` is a **different** strip —
20 tabs, `MFG.NOV.2024 EXP.OCT.2026`, no drug name in frame — kept as the negative case,
where the guardrail must decline. They are two products, not two shots of one strip
(`DEC-034`).

**Devanagari Read mode is tested on two photos, at every size in the sweep.**

`newspaper-marathi.png` is the real test — a photographed Marathi newspaper page, dense
printed body text, which is what Read mode exists for. `strip_paracip.jpg` is the hard
case: Cipla prints the brand name in Devanagari on the foil (`पॅरासिप-500`, beside each
`500 mg` box), so it is a few words on a curved reflective surface.

Both are read at 1280 *and* 1600, because the newspaper is where a downscale could
plausibly cost accuracy. Its long side is 1720, so unlike the strips it is genuinely
resized by both settings — 964×1280 against 1205×1600 — and newsprint body text is small
enough per glyph for that to matter. If accuracy drops at 1280 here, `max_side` really is
a safety parameter for Read mode, and that would be the evidence `DEC-030` asserted but
never had.

The run counts Devanagari codepoints rather than asking you to judge the output by eye:
PaddleOCR returns Latin happily even when the Devanagari recogniser contributes nothing,
so "it printed something" is not evidence that `lang='mr'` worked.

In [5]:
SAMPLES = PROJECT / 'data' / 'samples'
photos = sorted(p for p in SAMPLES.iterdir() if p.suffix.lower() in ('.jpg', '.png'))

for i, p in enumerate(photos):
    print(f'  [{i}] {p.name}  ({p.stat().st_size/1e3:.0f} KB)')

STRIP = SAMPLES / 'strip_paracip.jpg'      # the good read
# Both Devanagari cases, read at every max_side in the sweep:
#   the newspaper is dense printed body text -- what Read mode is actually for, and small
#   enough per glyph that a downscale could plausibly cost accuracy (1720px long side, so
#   unlike the strips it IS resized by both 1280 and 1600);
#   the strip is the hard case -- पॅरासिप-500 on curved reflective foil, a few words only.
DEVANAGARI = [SAMPLES / 'newspaper-marathi.png',
              SAMPLES / 'strip_paracip.jpg']

if not STRIP.exists():
    raise SystemExit(f'{STRIP} missing -- is data/samples/ present in the project?')

print('\nstrip     :', STRIP.name)
print('devanagari:', ', '.join(p.name for p in DEVANAGARI) if DEVANAGARI
      else '(none set -- §5 will skip)')
for _p in DEVANAGARI or []:
    if not _p.exists():
        raise SystemExit(f'{_p} missing -- is it committed and pushed?')

  [0] strip_paracip.jpg  (320 KB)
  [1] strip_partial.jpg  (308 KB)

strip     : strip_paracip.jpg
devanagari: (none set -- §6 will skip)


## 4. Medicine mode — the guardrail, end to end

OCR reads the strip, the drug name is matched against the verified database, expiry and MRP
are parsed. If OCR cannot produce a verified name the mode **declines** rather than guessing
(`DEC-007`).

**This section also produces the latency numbers RISK-1 needs.** It sweeps `max_side` over
1280 and 1600 and runs each setting twice, because the single figure recorded so far (42.9 s)
answers none of the questions actually being asked:

- it was measured at **1280**, while the build plan compared it to a <8 s target as if it
  were the default;
- 1600 is a **no-op on these fixtures** (they are 1600x1204), so the two settings differ by
  whether document unwarping runs on 1.9 MP or 1.2 MP;
- it bundled **one-time model loading** into the per-photo cost, which is the number that
  matters to a blind user holding a strip.

`max_side` is applied in `PaddleOCREngine._prepare()`, at inference time, so **one loaded
model serves both settings** — the load is paid and timed once, and the comparison isolates
preprocessing rather than re-measuring startup twice.

Both settings must return the same drug name, expiry and MRP. If they diverge, that is a
finding, not a latency measurement — stop and record it.

In [ ]:
%%writefile /content/ocr_phase.py
"""PaddlePaddle-only phase, run as its own process.

The crash that cost three days, and what the faulthandler stack finally showed:

    File "paddle/base/core.py", line 267 in <module>      <- SIGSEGV here
      ...
    File "paddlex/utils/import_guard.py", line 36 in import_paddle_module
    File "paddleocr/_common_args.py", line 105 in prepare_common_init_args
    File "app/engines/paddle_ocr.py", line 108 in _load

It died *importing libpaddle*, before any image was touched -- which is why --fast and
--max-side changed nothing. Both are inference-time knobs; this is a load-time crash.

Loaded extension modules at the moment of death included torch._C and TensorFlow, even
though this script imports neither: paddleocr defers loading paddle to paddlex, and
paddlex pulls torch and TF in first. Paddle, torch and TF each link their own
glog/gflags/OpenMP, and whichever initializes last dies in its static initializers. Same
conflict as DEC-006, but fired from inside PaddleOCR's own dependency chain.

Fix: import paddle first, into a pristine process. It is the first heavy import below,
and app/engines/paddle_ocr.py does the same in _load() so the laptop and web paths get it
too -- that copy needs a `git push` to reach this notebook, this one does not.

The instrumentation stays. It is what turned three days of guessing into one stack trace:
faulthandler before every import, mark() at each stage, unbuffered output (-u) so nothing
is lost when a process is killed by a signal.
"""
import faulthandler

faulthandler.enable()  # must precede every other import to be useful

# THE FIX -- keep this above every other heavy import. See the module docstring.
import paddle  # noqa: E402,F401

import argparse  # noqa: E402
import json  # noqa: E402
import os  # noqa: E402
import re  # noqa: E402
import resource  # noqa: E402
import sys  # noqa: E402
import time  # noqa: E402
from pathlib import Path  # noqa: E402

PROJECT = Path('/content/drishti')
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

HEAVY = ('paddle', 'torch', 'tensorflow', 'transformers', 'paddleocr', 'cv2', 'numpy')
_OMP = re.compile(r'/([^/\s]*(?:libgomp|libiomp5|libomp|libmkl_rt|libopenblas)[^/\s]*\.so[^\s]*)')


def native_runtimes() -> list[str]:
    """OpenMP/BLAS shared objects mapped into this process."""
    try:
        return sorted({m.group(1) for m in map(_OMP.search, open('/proc/self/maps')) if m})
    except OSError:
        return ['(no /proc)']


def peak_gb() -> float:
    """Peak RSS of this process. ru_maxrss is in kilobytes on Linux."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6


def mark(label: str) -> None:
    loaded = [m for m in HEAVY if m in sys.modules]
    print(f'[{label}] rss={peak_gb():.2f}GB loaded={loaded}', flush=True)


# paddle present and torch absent is the ordering this file exists to guarantee.
mark('start -- paddle imported first')
assert 'paddle' in sys.modules, 'paddle must be imported before anything else'
if 'torch' in sys.modules:
    print('WARNING: torch loaded before paddle -- the segfault is likely back', flush=True)

from app.drug_db import DrugDatabase  # noqa: E402
from app.engines.paddle_ocr import PaddleOCREngine  # noqa: E402
from app.modes.medicine import run as run_medicine  # noqa: E402
from app.modes.read import run as run_read  # noqa: E402
mark('app imported')

parser = argparse.ArgumentParser()
parser.add_argument('--strip', required=True, type=Path)
parser.add_argument('--devanagari', default=None,
                    help='comma-separated photos to read with lang=mr. Each is read at '
                         'every size in --max-side, because dense small text is where a '
                         'downscale could actually cost accuracy')
parser.add_argument('--out', required=True, type=Path)
parser.add_argument('--max-side', default='1280',
                    help='comma-separated sizes to sweep, e.g. "1280,1600". The FIRST is '
                         'the primary -- its result fills the checkpoint the rest of the '
                         'notebook reads. 1600 is a no-op on the 1600x1204 fixtures, so '
                         'unwarping runs full-size there (DEC-034)')
parser.add_argument('--repeats', type=int, default=1,
                    help='runs per size. The 2026-08-10 run measured 46.5/54.5s at 1280 '
                         'and 53.7/52.8s at 1600: pass-to-pass variance already exceeds '
                         'the gap between sizes, so there is no warm-up left to measure')
parser.add_argument('--phase', choices=('medicine', 'devanagari'), required=True,
                    help='ONE phase per process. Each PaddleOCR language is a complete '
                         'pipeline, and mr resolves to the server det model; holding en '
                         'open while loading mr was SIGKILLed at ~4 GB on 2026-08-10')
parser.add_argument('--fast', action='store_true',
                    help='disable document preprocessing. DEC-004 measured this destroying '
                         'accuracy, so a run needing it is a diagnosis, not a pass')
args = parser.parse_args()
print(f'max_side={args.max_side}  fast={args.fast}', flush=True)

sizes = [int(s) for s in args.max_side.split(',') if s.strip()]
primary = sizes[0]

# Results accumulate across phases. Each process loads whatever is already on disk, adds
# its own keys and writes the file back, so the phases compose without sharing memory.
checkpoint = {}
if args.out.exists():
    checkpoint = json.loads(args.out.read_text(encoding='utf-8'))


def save() -> None:
    """Persist now, not at the end of the script.

    The 2026-08-10 run computed a correct medicine result, then was SIGKILLed during the
    Devanagari phase before writing anything -- five minutes of OCR lost to a crash in
    unrelated work. Writing after each phase makes progress survive the next surprise.
    ensure_ascii=False keeps Devanagari readable in the file instead of \\uXXXX escapes.
    """
    checkpoint['peak_rss_gb'] = round(peak_gb(), 2)
    checkpoint['omp_runtimes'] = native_runtimes()
    args.out.write_text(json.dumps(checkpoint, ensure_ascii=False), encoding='utf-8')
    print(f'\ncheckpoint written: {args.out} (peak RSS {peak_gb():.2f} GB)', flush=True)


DEVANAGARI_RANGE = range(0x0900, 0x0980)


def devanagari_chars(text: str) -> int:
    """How much of the output is actually Devanagari.

    An objective signal beats eyeballing: PaddleOCR returns Latin happily when the
    Devanagari recogniser contributes nothing, so 'it printed something' is not evidence
    that lang=mr worked -- and nobody on this project can spot that by glancing at script
    they cannot read.
    """
    return sum(ord(ch) in DEVANAGARI_RANGE for ch in text)


def medicine_phase() -> None:
    """Sweep max_side on the strip, in a process that holds only the `en` pipeline."""
    engine = PaddleOCREngine(lang='en', max_side=primary, fast=args.fast)
    t0 = time.time()
    engine._load()
    load_seconds = time.time() - t0
    mark('paddleocr loaded -- past the segfault')
    print(f'model load: {load_seconds:.1f}s (one-time, not per photo)', flush=True)

    db = DrugDatabase.from_file()
    latency = {'load_seconds': round(load_seconds, 1), 'sizes': {}}
    results = {}

    for size in sizes:
        engine.max_side = size          # inference-time only; no reload
        passes = []
        for run in range(args.repeats):
            t0 = time.time()
            results[size] = run_medicine(args.strip, engine, db)
            passes.append(round(time.time() - t0, 1))
            print(f'  max_side={size}  pass {run + 1}/{args.repeats}: {passes[-1]}s',
                  flush=True)
        latency['sizes'][str(size)] = {'passes': passes, 'first_seconds': passes[0],
                                       'warm_seconds': min(passes)}

    result = results[primary]

    # Same strip, same model -- only the downscale differs. A disagreement is a finding.
    answers = {s: (r.drug_name, r.expiry_raw, r.mrp) for s, r in results.items()}
    if len(set(answers.values())) > 1:
        print('\n*** max_side CHANGED THE ANSWER -- this is a finding, not a timing run ***')
        for s, a in answers.items():
            print(f'    {s}: drug={a[0]!r} expiry={a[1]!r} mrp={a[2]!r}')
        print('    Record it in section 8 before drawing any latency conclusion.', flush=True)
    else:
        print(f'\nall sizes agree: {answers[primary]}', flush=True)

    warm = latency['sizes'][str(primary)]['warm_seconds']
    print(f'\n--- medicine mode  (warm {warm:.1f}s @ max_side={primary}, '
          f'peak RSS {peak_gb():.2f} GB) ---')
    print('verified :', result.ok)
    print('drug     :', result.drug_name)
    print('expiry   :', result.expiry_raw, '| expired:', result.expired)
    print('MRP      :', result.mrp)
    print('SPOKEN   :', result.message_en, flush=True)

    if not result.ok:
        print('\nDeclined. Either OCR missed the name, or it is absent from')
        print('data/drug_names_nlem2022.txt -- the 391 generic names of NLEM 2022 (DEC-032).')
        print('A brand-only strip with no generic printed on it is a legitimate decline.')
        print('--fast disables the preprocessing DEC-004 found load-bearing; leave it off.')

    checkpoint.update({
        'ok': result.ok,
        'drug_name': result.drug_name,
        'expiry_raw': result.expiry_raw,
        'expired': result.expired,
        'mrp': result.mrp,
        'message_en': result.message_en,
        'medicine_seconds': warm,
        'latency': latency,
        'max_side': primary,
        'fast': args.fast,
    })


def devanagari_phase() -> None:
    """Read every photo at every size with lang=mr, in a process of its own.

    Its own process is not tidiness. Each PaddleOCR language is a full pipeline --
    doc_ori + UVDoc + textline_ori + det + rec -- and `mr` resolves to the *server* det
    model, not a mobile one. Holding `en` open while loading `mr` peaked past 4 GB and was
    SIGKILLed on 2026-08-10. Sequential processes cost one extra model load and cannot
    collide.
    """
    if not args.devanagari:
        print('\nNo Devanagari photo passed -- skipping. Read mode in Marathi stays '
              'unverified.')
        return

    photos = [Path(p) for p in args.devanagari.split(',') if p.strip()]
    absent = [str(p) for p in photos if not p.exists()]
    if absent:
        raise SystemExit(f'Devanagari photos not found: {absent}')

    engine = PaddleOCREngine(lang='mr', max_side=primary, fast=args.fast)
    t0 = time.time()
    engine._load()
    mark('devanagari pipeline loaded')
    print(f'devanagari model load: {time.time() - t0:.1f}s (one-time)', flush=True)

    runs = []
    for photo in photos:
        for size in sizes:
            engine.max_side = size
            t0 = time.time()
            text = run_read(photo, engine)
            seconds = round(time.time() - t0, 1)
            run = {'image': photo.name, 'max_side': size, 'seconds': seconds,
                   'chars': len(text), 'devanagari_chars': devanagari_chars(text),
                   'text': text}
            runs.append(run)
            print(f"\n--- devanagari: {photo.name} @ max_side={size} "
                  f"({seconds}s, {run['devanagari_chars']} devanagari chars "
                  f"of {run['chars']} total) ---")
            print(text[:600] + ('...' if len(text) > 600 else ''), flush=True)
            # Persist per read: a photo that survives is banked even if the next one dies.
            checkpoint['devanagari_runs'] = runs
            save()

    print('\n--- devanagari summary ---')
    print(f"{'image':26} | {'max_side':>8} | {'deva':>5} | {'total':>5} | {'secs':>5}")
    print('-' * 66)
    for r in runs:
        print(f"{r['image'][:26]:26} | {r['max_side']:>8} | "
              f"{r['devanagari_chars']:>5} | {r['chars']:>5} | {r['seconds']:>5}")

    best = max(runs, key=lambda r: r['devanagari_chars'])
    if best['devanagari_chars'] == 0:
        print('\n*** NO DEVANAGARI RECOGNIZED IN ANY RUN ***')
        print('    lang=mr loaded but returned no Devanagari codepoints at all. That is')
        print('    the RISK-7 failure -- record it, and DEC-005 needs a caveat.', flush=True)
    else:
        print(f"\nbest: {best['image']} @ {best['max_side']} "
              f"-- {best['devanagari_chars']} devanagari chars", flush=True)

    primary_runs = [r for r in runs if r['max_side'] == primary]
    if primary_runs:
        checkpoint['devanagari_text'] = primary_runs[0]['text']
        checkpoint['devanagari_seconds'] = primary_runs[0]['seconds']


if args.phase == 'medicine':
    medicine_phase()
else:
    devanagari_phase()

save()


In [ ]:
import json
from pathlib import Path

CHECKPOINT = Path('/content/drishti_checkpoint.json')

MAX_SIDES = (1280, 1600)   # first is primary; both are swept
REPEATS = 1                # see below -- the warm/cold question is already answered
FAST = False               # leave OFF: DEC-004 measured it costing the drug name, expiry, MRP

# Why two subprocesses instead of one.
#
# Each PaddleOCR language is a complete pipeline -- doc_ori + UVDoc + textline_ori + det +
# rec -- and `mr` resolves to PP-OCRv5_server_det, not a mobile model. The 2026-08-10 run
# held the `en` pipeline open while loading `mr`, passed 3.9 GB, and was SIGKILLed partway
# through the newspaper sweep. Worse, it had already produced a correct medicine result:
# the checkpoint was only written at the end, so the crash destroyed five minutes of OCR
# that had nothing to do with Devanagari.
#
# Now each phase is its own process and saves as it goes. The cost is one extra model
# load; the benefit is that neither phase can lose the other's work.
#
# Why REPEATS = 1: that run measured 46.5s then 54.5s at 1280, and 53.7s then 52.8s at
# 1600. Pass-to-pass variance is larger than the gap between sizes, so a second pass buys
# nothing but four more minutes.

SIGNALS = {-11: 'SIGSEGV. If the stack ends in paddle/base/core.py the import-order fix did '
                'not take -- check that ocr_phase.py still imports paddle first, and that '
                'no [stage] line lists torch before paddle.',
           -9:  'SIGKILL -- the OOM killer. With the phases split this should not recur; if '
                'it does, drop MAX_SIDES to a single size (1024, then 896).',
           -6:  'SIGABRT -- a library aborted deliberately, usually a failed internal check.'}


def available_gb() -> float:
    for line in Path('/proc/meminfo').read_text().splitlines():
        if line.startswith('MemAvailable:'):
            return int(line.split()[1]) / 1e6
    return float('nan')


# A stale checkpoint from an earlier run would masquerade as this run's result.
if CHECKPOINT.exists():
    CHECKPOINT.unlink()

SIZES = ','.join(str(s) for s in MAX_SIDES)
FAST_FLAG = ' --fast' if FAST else ''
BASE = f'python -u /content/ocr_phase.py --out "{CHECKPOINT}" --max-side {SIZES}'

print(f'RAM available before OCR: {available_gb():.1f} GB\n')

# -u is not cosmetic: stdout is block-buffered when piped, so a process killed by a signal
# loses everything it printed. That is why the first crash reported almost nothing.
cmd = f'{BASE} --phase medicine --strip "{STRIP}" --repeats {REPEATS}{FAST_FLAG}'
print('$', cmd, '\n')
!{cmd}

if _exit_code != 0:
    raise SystemExit(f'medicine phase exited {_exit_code}: '
                     + SIGNALS.get(_exit_code, 'a Python error -- traceback is above.'))

print(f'\n{"=" * 70}\nmedicine phase done and saved. RAM now: {available_gb():.1f} GB\n')

if DEVANAGARI:
    photos = ','.join(str(p) for p in DEVANAGARI)
    cmd = f'{BASE} --phase devanagari --strip "{STRIP}" --devanagari "{photos}"{FAST_FLAG}'
    print('$', cmd, '\n')
    !{cmd}

    # Deliberately not fatal. Medicine results are already on disk and §6-§7 depend only
    # on those, so a Devanagari failure should cost the Devanagari row -- not the run.
    if _exit_code != 0:
        print(f'\nWARNING: devanagari phase exited {_exit_code}: '
              + SIGNALS.get(_exit_code, 'a Python error -- traceback is above.'))
        print('Medicine results are saved; §6-§7 can still run. RISK-7 stays open.')

checkpoint = json.loads(CHECKPOINT.read_text(encoding='utf-8'))
print(f"\nsurvived. peak RSS {checkpoint['peak_rss_gb']} GB at max_side={checkpoint['max_side']}"
      f", fast={checkpoint['fast']}")

# The table to copy into section 8. Model load is listed separately from the per-photo
# cost on purpose: RISK-1's <8s target is about the second, and quoting a cold figure
# against it understates the system by the whole startup time.
lat = checkpoint['latency']
print(f"\nmodel load (one-time): {lat['load_seconds']}s\n")
print(f"{'max_side':>10} | {'first':>7} | {'warm':>7} | all passes")
print('-' * 52)
for size, row in lat['sizes'].items():
    print(f"{size:>10} | {row['first_seconds']:>6.1f}s | {row['warm_seconds']:>6.1f}s | "
          f"{row['passes']}")
print('\n<8s target (RISK-1): warm per-photo OCR is the number to compare.')


## 5. Read mode — Devanagari

Moved up next to medicine mode on purpose: both use **PaddleOCR only**, so they belong in
the same PaddlePaddle-only phase (see the restart notice below).

`lang='mr'` resolves to `devanagari_PP-OCRv5_mobile_rec`. The code path is confirmed; what
has never been tested is the model against actual Devanagari text — until this run.

**What counts as a pass**, per photo:

- **newspaper** — substantial Devanagari, hundreds of codepoints. The headline
  (`खंक तिजोरीमुळे भिवंडी भकास`) is large and clean and should come back nearly intact;
  body text is the harder ask. Compare the two `max_side` rows before concluding anything
  about the model — a low count at 1280 and a high one at 1600 is a downscale problem, not
  a recogniser problem.
- **strip** — a handful of Devanagari characters, `पॅरासिप` at best. Thin is expected here;
  zero is informative but not fatal, since foil is the worst case in the whole project.

**Zero Devanagari across *both* photos at *both* sizes** is the `RISK-7` failure: it would
mean `devanagari_PP-OCRv5_mobile_rec` is not usable, `DEC-005` needs a caveat, and Read
mode ships English-only for Sem-7. The run prints that verdict explicitly rather than
leaving it to be inferred from output you cannot read.

Mixed Latin and Devanagari is normal — the model is not restricted to one script, and the
newspaper page carries an email address and a masthead in Latin.

In [ ]:
# Read mode already ran, inside the subprocess above -- it is PaddleOCR too, so it belongs
# in that process rather than a second one. Nothing to load here, just the results.
runs = checkpoint.get('devanagari_runs') or []
if not runs:
    print('No Devanagari photo set in §3 -- skipped. Read mode in Marathi stays unverified.')
else:
    print(f"{'image':26} | {'max_side':>8} | {'deva':>5} | {'total':>5} | {'secs':>5}")
    print('-' * 66)
    for r in runs:
        print(f"{r['image'][:26]:26} | {r['max_side']:>8} | "
              f"{r['devanagari_chars']:>5} | {r['chars']:>5} | {r['seconds']:>5}")

    best = max(runs, key=lambda r: r['devanagari_chars'])
    if best['devanagari_chars'] == 0:
        print('\nNO DEVANAGARI RECOGNIZED -- this is the RISK-7 failure. Record it in §8:')
        print('DEC-005 claims lang=mr resolves to a working Devanagari model; it resolves,')
        print('but reads nothing. Read mode ships English-only for Sem-7.')
    else:
        print(f"\n--- best: {best['image']} @ max_side={best['max_side']} "
              f"({best['devanagari_chars']} devanagari chars) ---")
        print(best['text'])

    # Same photo at two sizes is the accuracy question DEC-030 asserted without evidence.
    for image in dict.fromkeys(r['image'] for r in runs):
        counts = {r['max_side']: r['devanagari_chars'] for r in runs if r['image'] == image}
        if len(counts) > 1:
            lo, hi = min(counts), max(counts)
            delta = counts[hi] - counts[lo]
            verdict = ('downscaling COSTS accuracy here' if delta > 0 else
                       'no accuracy cost from downscaling' if delta == 0 else
                       'downscaling HELPED -- unexpected, note it')
            print(f'\n{image}: {counts[lo]} chars @ {lo} vs {counts[hi]} @ {hi} '
                  f'-> {verdict}')

## No restart needed

Earlier versions asked you to `Runtime → Restart session` here. That never helped: the
failures were an out-of-memory kernel and then a segfault importing `libpaddle`, neither of
which a restart placed *after* the crash could prevent.

§4 now runs OCR in a subprocess that imports paddle before anything else. Paddle never
enters this kernel at all, so §6–§7 follow straight on.

**Run the bootstrap cell below** — it defines `STRIP` and `checkpoint` for §6–§7 and is safe
to run whether or not the kernel was restarted.

In [ ]:
# Idempotent bootstrap for the torch phase. Safe to run whether or not the kernel was
# restarted -- it only restores names, and the VM's disk (packages, the clone, the
# checkpoint) survives a restart even though Python's in-memory state does not.
import json
import os
import sys
import time
from pathlib import Path

PROJECT = Path('/content/drishti')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# §3 may have been wiped by a restart, and §7 needs STRIP -- re-derive rather than re-running
# §3, which would be harmless now but is one more thing to remember.
SAMPLES = PROJECT / 'data' / 'samples'
STRIP = SAMPLES / 'strip_paracip.jpg'

CHECKPOINT = Path('/content/drishti_checkpoint.json')
if not CHECKPOINT.exists():
    raise SystemExit(
        f'{CHECKPOINT} not found. Did the OCR-phase cell (§4) finish? Scroll up and re-run it.'
    )

checkpoint = json.loads(CHECKPOINT.read_text(encoding='utf-8'))

# The version check §2 used to do. It lives here because importing transformers is what put
# torch and TensorFlow in the kernel during the OCR phase -- from this cell on, that is
# exactly what we want loaded. The sys.modules purge matters only if an earlier cell in this
# same kernel imported transformers before the pip downgrade took effect.
for _mod in [m for m in sys.modules if m == 'transformers' or m.startswith('transformers.')]:
    del sys.modules[_mod]
import transformers

print('transformers :', transformers.__version__, '(must be 4.x for IndicTransToolkit)')
print('project root :', PROJECT)
print('strip        :', STRIP.name, '(exists)' if STRIP.exists() else '(MISSING)')
print('checkpoint   :', checkpoint)

In [ ]:
# Preflight: can this runtime actually fetch every model §6-§7 needs?
#
# ai4bharat gated the IndicTrans2 repo, so an unauthenticated runtime fails with a 401 --
# surfaced by transformers as a 22-frame traceback ending in a bare OSError, which reads
# like a network fault rather than "you need to accept a licence". Checking all four repos
# up front costs seconds and tells you every missing permission at once, instead of one
# failure per cell with a model download in between.
from huggingface_hub import HfApi
from huggingface_hub.errors import GatedRepoError, RepositoryNotFoundError

REPOS = [
    ('ai4bharat/indictrans2-en-indic-dist-200M', 'translation (§6)'),
    ('facebook/mms-tts-mar', 'Marathi speech (§6)'),
    ('facebook/mms-tts-hin', 'Hindi speech (§6)'),
    ('HuggingFaceTB/SmolVLM-Instruct', 'scene + ask (§7)'),
]

api = HfApi()
blocked = []
for repo, why in REPOS:
    try:
        api.model_info(repo)
        print(f'  ok      {repo}  -- {why}')
    except GatedRepoError:
        blocked.append(repo)
        print(f'  GATED   {repo}  -- {why}')
    except RepositoryNotFoundError:
        # A gated repo looks like a missing one when you are not authenticated at all.
        blocked.append(repo)
        print(f'  404     {repo}  -- {why} (gated, or genuinely renamed)')

if blocked:
    print('\nBlocked. For each repo above:')
    for repo in blocked:
        print(f'  1. Accept the licence at https://huggingface.co/{repo}')
    print('  2. Create a READ token at https://huggingface.co/settings/tokens')
    print('  3. Colab sidebar -> Secrets (key icon) -> add HF_TOKEN -> enable notebook access')
    print('  4. Re-run this cell. No restart needed; nothing above is lost.')
    raise SystemExit(f'{len(blocked)} model(s) not accessible -- see the steps above')

print('\nAll models reachable. The downloads are one-time; everything runs offline after.')

## 6. Marathi output and speech — the Phase-1 exit criterion

Reads `message_en` from the checkpoint the OCR subprocess wrote, rather than from a Python
variable — the OCR result was produced in a different process, so there is no variable to
read.

In [ ]:
from IPython.display import Audio, display

from app.engines.indictrans import IndicTrans2Translator
from app.engines.mms_tts import MMSTTSEngine
from app.speech import deliver

if not checkpoint.get('message_en'):
    raise SystemExit('checkpoint has no message_en -- did medicine mode (§4) succeed?')

translator = IndicTrans2Translator()
tts = MMSTTSEngine(out_dir=Path('/content/audio'))

for lang in ('mr', 'hi'):
    t0 = time.time()
    spoken = deliver(checkpoint['message_en'], lang=lang, translator=translator,
                     tts=tts, speak=True)
    print(f'--- {lang} ({time.time()-t0:.1f}s) ---')
    print(spoken.text_out)
    display(Audio(str(spoken.audio_path)))

## 7. Scene mode — the VLM

Same PyTorch-only session as §6 — no second restart needed, SmolVLM and IndicTransToolkit
coexist fine since both are PyTorch.

In [ ]:
from app.engines.smolvlm import SmolVLMEngine
from app.modes.ask import run as run_ask
from app.modes.scene import run as run_scene

vlm = SmolVLMEngine()

t0 = time.time()
print('--- scene mode ---')
print(run_scene(STRIP, vlm), f'({time.time()-t0:.1f}s)')

t0 = time.time()
print('\n--- ask mode ---')
print(run_ask(STRIP, vlm, 'what is written on this?'), f'({time.time()-t0:.1f}s)')

## 8. Findings — run of 2026-08-10, Colab **CPU-only** (no GPU allocated)

**Phase-1 exit criteria met.** A photographed medicine strip produced a spoken Marathi
answer through the real `app/` code; Devanagari Read mode was verified on real newsprint;
and every mode ran end to end in one session.

`nvidia-smi: command not found` — this runtime had **no GPU**, so every VLM number below is
a CPU number. That is the honest laptop figure and the one the project should quote, but it
is not comparable to the 1.2 s/answer measured on a T4 in notebook 01.

| Check | Result | Latency |
|---|---|---|
| Medicine: drug name verified | ✅ `Paracetamol`, matched against the database | 56.2 s @ 1280 |
| Medicine: expiry parsed | ✅ `APR.28` — correct, and the only expiry on this strip | |
| Medicine: MRP parsed | ✅ `10.30` | |
| Both `max_side` settings agree | ✅ identical drug name, expiry and MRP | |
| OCR subprocess survived | ✅ medicine phase peaked 3.56 GB; **Devanagari phase peaked 11.65 GB** | |
| Devanagari Read mode | ✅ **1010 Devanagari chars of 1238** on newsprint | 69.4 s |
| Marathi translation readable | ✅ `हे पॅरासिटामॉल आहे. हे APR.28 पर्यंत वैध आहे. एमआरपी 10.30 रुपये आहे.` | |
| Hindi translation readable | ✅ `यह पेरासिटामोल है। यह APR.28 तक मान्य है।` | |
| Marathi / Hindi speech | 🟡 audio produced; **not yet judged by a Marathi speaker** | |
| Scene mode | ✅ **full descriptive paragraph** — `DEC-031` confirmed | 481.3 s (CPU) |
| Ask mode | ✅ answered `Paracip-500` — correct, but see below | 314.8 s (CPU) |

### `max_side` **is** a latency lever, after all — correcting `DEC-036`

Measured in one session, on one loaded model:

| | `max_side=1280` | `max_side=1600` | Same answer? |
|---|---|---|---|
| Medicine (strip) | **56.2 s** | 76.1 s | ✅ identical |
| Read (newspaper) | **69.4 s** | 103.6 s | ✅ 1010 Devanagari chars both |
| Read (strip foil) | 58.0 s | 76.6 s | 14 vs 12 chars — noise |

1280 is **25–35% faster with no measurable accuracy cost**, including on the newspaper,
which is the fixture most likely to suffer from a downscale. `DEC-036` concluded the
opposite from the previous session's numbers (46.5/54.5 s at 1280 against 53.7/52.8 s at
1600) — but those were two passes on a differently-loaded Colab machine, and the spread
within a single setting was as large as the gap between settings. This run repeats the
comparison back-to-back in one process and the gap is consistent across all three photos.

**The method lesson, which belongs in the report:** the same knob has now been called
dangerous (`DEC-030`), then irrelevant (`DEC-036`), then useful — and only the third
reading came from a controlled comparison. Cross-session Colab timings are not evidence;
within-session, same-process, same-model comparisons are.

### Scene mode works — and confabulates

`describe()` returned prose where the old code returned `Paracip-500`, so `DEC-031` is
confirmed against the real model. But read what it said:

> The image displays a blister pack of Paracetamol tablets… labeled as "Paracetamol Tablets
> 500mg" and **contains 30 tablets**… made of a **clear plastic material** and has a **white
> backing**… The back… contains the manufacturer's information, the expiry date, and the
> batch number.

The strip holds **10** tablets (`Rs.10.30 FOR 10 TABS`) and is opaque foil, not clear
plastic with a white backing. The drug name and dosage are right; the specifics are
invented, fluently and in the same confident register as the true parts.

This is `DEC-007` demonstrated rather than argued. A blind user cannot check "30 tablets"
against the object in their hand. Scene mode is for orientation, and every fact a user
might act on — drug name, expiry, MRP — must keep coming from OCR plus the database.
Worth quoting verbatim in the report: it is more convincing than the Moondream anecdote
`DEC-007` currently rests on, because this is the shipped model on a real fixture.

### Ask mode answered this time — the earlier abstention was not a law

Asked "what is written on this?", ask mode returned `Paracip-500`, which is correct. The
previous run abstained on the same photo with the same prompt, and §8 recorded that
abstention as direct evidence for `DEC-012`.

Both behaviours are acceptable and the model is simply not deterministic here, but the
earlier write-up over-claimed: one sample is an anecdote, not a validation. `DEC-012` does
not need it — it rests on the VizWiz failures (`545` vs `1545`, `Twelve years` vs `dog
years`), which are measured over 500 samples. The anecdote has been withdrawn from the
justification rather than replaced with the opposite anecdote.

### Latency is the open problem, and it is worse on CPU than documented

- **OCR ~56 s** per photo at 1280, plus a one-time 59.2 s model load
- **Translation + TTS** comfortable
- **VLM: 481 s scene, 315 s ask** — on CPU

The README says CPU inference takes "tens of seconds rather than the 1.2 s measured on a
Colab T4". It takes **five to eight minutes**. That claim needs correcting before anyone
plans a demo around it.

Against the <8 s target (`RISK-1`), the levers now ranked by evidence:

1. **`max_side=1280`** — measured, free, already applied. 25–35%.
2. **Mobile model tier** — `en` loads `PP-OCRv6_medium_det`/`_rec`, `mr` loads
   `PP-OCRv5_server_det`. None are the mobile variants the Android target needs anyway.
3. **A GPU for the demo, or no VLM in the demo path.** Scene and Ask are the modes blowing
   the budget; medicine, read and currency do not touch the VLM. If the review demo is
   CPU-only, drive it with the OCR modes and show scene/ask as a recorded clip.

### Memory: the phase split was load-bearing

The Devanagari phase peaked at **11.65 GB**, against 10.7 GB free at the start and 3.56 GB
for the medicine phase. Running both languages in one process was never going to fit, and
the previous session's SIGKILL was not bad luck. `DEC-035` stands, and `MAX_SIDES` should
not grow a third entry without re-checking headroom.

### Three native runtimes coexist happily once load order is right

`libgomp`, `libiomp5` and `libopenblas` were all mapped into the OCR process simultaneously.
The conflict in `DEC-006`/`DEC-027` was never about avoiding them, only about importing
paddle first.
